# Cylinder Flow — Mixed-Form PINN (TF2)

Steady 2D laminar flow past a cylinder in a channel.
Implements the mixed-form physics-informed neural network from:

> Rao, Sun & Liu (2020). *Physics-informed deep learning for incompressible laminar flows.*
> TAML. [arXiv:2002.10558](https://arxiv.org/abs/2002.10558)

**Physics:** Incompressible Navier-Stokes (steady), stream-function formulation.  
**Network outputs:** ψ, p, s₁₁, s₂₂, s₁₂  →  u = ∂ψ/∂y, v = −∂ψ/∂x  
**Optimiser:** Adam (10 k iters) → L-BFGS-B (up to 50 k iters per round)

## 1. Imports

In [ ]:
import numpy as np
import time
import pickle
import scipy.io
import scipy.optimize
import matplotlib.pyplot as plt
import tensorflow as tf

tf.random.set_seed(1234)
np.random.seed(1234)

## 2. Sampling & Helpers

In [ ]:
def lhs(n_dim, n_samples, seed=None):
    """Latin Hypercube Sampling in [0,1]^n_dim."""
    rng = np.random.default_rng(seed)
    cut = np.linspace(0.0, 1.0, n_samples + 1)
    u = rng.random((n_samples, n_dim))
    a, b = cut[:n_samples], cut[1:n_samples + 1]
    rdpoints = a[:, None] + (b - a)[:, None] * u
    H = np.empty_like(rdpoints)
    for j in range(n_dim):
        H[:, j] = rdpoints[rng.permutation(n_samples), j]
    return H


def DelCylPT(XY_c, xc=0.2, yc=0.2, r=0.05):
    """Remove points inside the cylinder."""
    dst = np.sqrt((XY_c[:,0]-xc)**2 + (XY_c[:,1]-yc)**2)
    return XY_c[dst > r, :]


def preprocess_mat(dir_path):
    """Load reference solution (ANSYS Fluent .mat)."""
    data = scipy.io.loadmat(dir_path)
    X, Y = data['x'], data['y']
    vx, vy, P = data['vx'], data['vy'], data['p']
    return (X.flatten()[:,None], Y.flatten()[:,None],
            vx.flatten()[:,None], vy.flatten()[:,None],
            P.flatten()[:,None])


def postProcess(xmin, xmax, ymin, ymax, field_ref, field_pinn,
                s=2, alpha=0.5, marker='o'):
    """6-panel scatter: PINN (left) vs Reference (right)."""
    x_r, y_r, u_r, v_r, p_r = field_ref
    x_p, y_p, u_p, v_p, p_p = field_pinn
    fig, ax = plt.subplots(3, 2, figsize=(7, 4))
    fig.subplots_adjust(hspace=0.2, wspace=0.2)
    def _sc(axh, x, y, c, title, vmin=None, vmax=None):
        cf = axh.scatter(x, y, c=c, alpha=alpha, edgecolors='none',
                         cmap='rainbow', marker=marker, s=int(s),
                         vmin=vmin, vmax=vmax)
        axh.set_aspect('equal'); axh.axis('off')
        axh.set_xlim([xmin,xmax]); axh.set_ylim([ymin,ymax])
        axh.set_title(title, fontsize=8)
        fig.colorbar(cf, ax=axh, fraction=0.046, pad=0.04)
    _sc(ax[0,0], x_p, y_p, u_p, r'$u$ PINN')
    _sc(ax[1,0], x_p, y_p, v_p, r'$v$ PINN')
    _sc(ax[2,0], x_p, y_p, p_p, r'$p$ PINN', vmin=-0.25, vmax=4.0)
    _sc(ax[0,1], x_r, y_r, u_r, r'$u$ Ref')
    _sc(ax[1,1], x_r, y_r, v_r, r'$v$ Ref')
    _sc(ax[2,1], x_r, y_r, p_r, r'$p$ Ref',  vmin=-0.25, vmax=4.0)
    plt.tight_layout(); plt.show()

## 3. Model — MLP + Mixed-Form PINN

In [ ]:
# Helpers
# -------------------------
def DelCylPT(XY_c, xc=0.0, yc=0.0, r=0.1):
    """Delete points within cylinder."""
    dst = np.sqrt((XY_c[:,0]-xc)**2 + (XY_c[:,1]-yc)**2)
    return XY_c[dst > r, :]

def preprocess_mat(dir_path):
    """Load reference solution from Fenics or Fluent mat file.
       Expects keys: x,y,p,vx,vy"""
    data = scipy.io.loadmat(dir_path)
    X = data['x']; Y = data['y']
    P = data['p']
    vx = data['vx']; vy = data['vy']
    return (X.flatten()[:,None], Y.flatten()[:,None],
            vx.flatten()[:,None], vy.flatten()[:,None],
            P.flatten()[:,None])

def postProcess(xmin, xmax, ymin, ymax, field_ref, field_pinn, s=2, alpha=0.5, marker='o'):
    """Same plotting style as your original code."""
    x_ref, y_ref, u_ref, v_ref, p_ref = field_ref
    x_p, y_p, u_p, v_p, p_p = field_pinn

    fig, ax = plt.subplots(nrows=3, ncols=2, figsize=(7, 4))
    fig.subplots_adjust(hspace=0.2, wspace=0.2)

    def _scatter(axh, x, y, c, title, vmin=None, vmax=None):
        cf = axh.scatter(x, y, c=c, alpha=alpha, edgecolors='none',
                         cmap='rainbow', marker=marker, s=int(s),
                         vmin=vmin, vmax=vmax)
        axh.axis('square')
        for key, spine in axh.spines.items():
            spine.set_visible(False)
        axh.set_xticks([]); axh.set_yticks([])
        axh.set_xlim([xmin, xmax]); axh.set_ylim([ymin, ymax])
        axh.set_title(title)
        fig.colorbar(cf, ax=axh, fraction=0.046, pad=0.04)

    _scatter(ax[0,0], x_p, y_p, u_p, r'$u$ (PINN)')
    _scatter(ax[1,0], x_p, y_p, v_p, r'$v$ (PINN)')
    _scatter(ax[2,0], x_p, y_p, p_p, r'$p$ (PINN)', vmin=-0.25, vmax=4.0)

    _scatter(ax[0,1], x_ref, y_ref, u_ref, r'$u$ (REF)')
    _scatter(ax[1,1], x_ref, y_ref, v_ref, r'$v$ (REF)')
    _scatter(ax[2,1], x_ref, y_ref, p_ref, r'$p$ (REF)', vmin=-0.25, vmax=4.0)

    plt.show()

# -------------------------
# Keras network
# -------------------------
class MLP(tf.keras.Model):
    def __init__(self, layers_sizes, activation="tanh"):
        super().__init__()
        self.hidden = []
        for w in layers_sizes[:-1]:
            self.hidden.append(tf.keras.layers.Dense(
                w, activation=activation,
                kernel_initializer="glorot_normal"
            ))
        self.out = tf.keras.layers.Dense(
            layers_sizes[-1], activation=None,
            kernel_initializer="glorot_normal"
        )

    def call(self, x, training=False):
        z = x
        for lyr in self.hidden:
            z = lyr(z)
        return self.out(z)

# -------------------------
# Mixed-form PINN in TF2
# -------------------------
class PINN_laminar_flow_TF2:
    """
    TF2 rewrite of your TF1 class:
    Outputs: psi, p, s11, s22, s12
    u = dpsi/dy
    v = -dpsi/dx
    """
    def __init__(self, Collo, INLET, OUTLET, WALL, uv_layers, lb, ub,
                 rho=1.0, mu=0.02, bc_weight=2.0):

        self.lb = lb.astype(np.float32).reshape(1,2)
        self.ub = ub.astype(np.float32).reshape(1,2)

        self.rho = tf.constant(rho, dtype=tf.float32)
        self.mu  = tf.constant(mu, dtype=tf.float32)
        self.bc_weight = tf.constant(bc_weight, dtype=tf.float32)

        # Data
        self.Collo = Collo.astype(np.float32)
        self.INLET = INLET.astype(np.float32)   # [x,y,u,v]
        self.OUTLET = OUTLET.astype(np.float32) # [x,y]
        self.WALL = WALL.astype(np.float32)     # [x,y]

        # Model
        hidden_widths = uv_layers[1:-1]
        out_dim = uv_layers[-1]  # 5
        self.net = MLP(hidden_widths + [out_dim], activation="tanh")

        self.loss_rec   = []
        self.loss_total = []
        self.loss_pde   = []
        self.loss_wall  = []
        self.loss_inlet = []
        self.loss_outlet = []

    # --- forward outputs (psi,p,s11,s22,s12) ---
    def net_psips(self, x, y):
        xy = tf.concat([x, y], axis=1)
        out = self.net(xy)
        psi = out[:,0:1]
        p   = out[:,1:2]
        s11 = out[:,2:3]
        s22 = out[:,3:4]
        s12 = out[:,4:5]
        return psi, p, s11, s22, s12

    # --- compute u,v and first derivatives with a tape ---
    def uv_and_grads(self, x, y):
        with tf.GradientTape(persistent=True) as tape:
            tape.watch([x, y])
            psi, p, s11, s22, s12 = self.net_psips(x, y)
            u = tape.gradient(psi, y)
            v = -tape.gradient(psi, x)

        u_x = tape.gradient(u, x)
        u_y = tape.gradient(u, y)
        v_x = tape.gradient(v, x)
        v_y = tape.gradient(v, y)

        s11_x = tape.gradient(s11, x)
        s12_x = tape.gradient(s12, x)
        s12_y = tape.gradient(s12, y)
        s22_y = tape.gradient(s22, y)

        del tape
        return u, v, p, s11, s22, s12, u_x, u_y, v_x, v_y, s11_x, s12_x, s12_y, s22_y

    # --- PDE residuals (same as your net_f) ---
    def residuals(self, x, y):
        rho = self.rho
        mu  = self.mu

        (u, v, p, s11, s22, s12,
         u_x, u_y, v_x, v_y,
         s11_x, s12_x, s12_y, s22_y) = self.uv_and_grads(x, y)

        f_u = rho*(u*u_x + v*u_y) - s11_x - s12_y
        f_v = rho*(u*v_x + v*v_y) - s12_x - s22_y

        f_s11 = -p + 2.0*mu*u_x - s11
        f_s22 = -p + 2.0*mu*v_y - s22
        f_s12 = mu*(u_y + v_x) - s12

        f_p = p + 0.5*(s11 + s22)
        return f_u, f_v, f_s11, f_s22, f_s12, f_p

    # --- loss ---
    def loss_fn(self, Xc, WALL, INLET, OUTLET):
        # Collocation residuals
        xc = Xc[:,0:1]; yc = Xc[:,1:2]
        fu, fv, fs11, fs22, fs12, fp = self.residuals(xc, yc)
        loss_f = (tf.reduce_mean(tf.square(fu)) +
                  tf.reduce_mean(tf.square(fv)) +
                  tf.reduce_mean(tf.square(fs11)) +
                  tf.reduce_mean(tf.square(fs22)) +
                  tf.reduce_mean(tf.square(fs12)) +
                  tf.reduce_mean(tf.square(fp)))

        # WALL: u=v=0
        xw = WALL[:,0:1]; yw = WALL[:,1:2]
        u_w, v_w, *_ = self.uv_and_grads(xw, yw)
        loss_wall = tf.reduce_mean(tf.square(u_w)) + tf.reduce_mean(tf.square(v_w))

        # INLET: match prescribed u,v
        xi = INLET[:,0:1]; yi = INLET[:,1:2]
        ui = INLET[:,2:3]; vi = INLET[:,3:4]
        u_i, v_i, *_ = self.uv_and_grads(xi, yi)
        loss_in = tf.reduce_mean(tf.square(u_i - ui)) + tf.reduce_mean(tf.square(v_i - vi))

        # OUTLET: gauge pressure condition
        xo = OUTLET[:,0:1]; yo = OUTLET[:,1:2]
        _, p_o, _, _, _ = self.net_psips(xo, yo)

        # More stable than forcing p=0 pointwise:
        loss_out = tf.square(tf.reduce_mean(p_o))

        total = loss_f + self.bc_weight*(loss_wall + loss_in + loss_out)
        return total, loss_f, loss_wall, loss_in, loss_out

    # --- Adam training loop ---
    def train_adam(self, iters=10000, lr=5e-4, print_every=10):
        opt = tf.keras.optimizers.Adam(learning_rate=lr)

        Xc = tf.convert_to_tensor(self.Collo, dtype=tf.float32)
        WALL = tf.convert_to_tensor(self.WALL, dtype=tf.float32)
        INLET = tf.convert_to_tensor(self.INLET, dtype=tf.float32)
        OUTLET = tf.convert_to_tensor(self.OUTLET, dtype=tf.float32)

        for it in range(iters):
            with tf.GradientTape() as tape:
                loss, lf, lw, lin, lout = self.loss_fn(Xc, WALL, INLET, OUTLET)
            grads = tape.gradient(loss, self.net.trainable_variables)
            opt.apply_gradients(zip(grads, self.net.trainable_variables))

            if it % print_every == 0:
                print(f"It {it:6d} | loss={loss.numpy():.3e} "
                      f"| f={lf.numpy():.3e} wall={lw.numpy():.3e} "
                      f"in={lin.numpy():.3e} out={lout.numpy():.3e}")
            self.loss_rec.append(float(loss.numpy()))
            self.loss_total.append(float(loss.numpy()))
            self.loss_pde.append(float(lf.numpy()))
            self.loss_wall.append(float(lw.numpy()))
            self.loss_inlet.append(float(lin.numpy()))
            self.loss_outlet.append(float(lout.numpy()))

    # --- L-BFGS with SciPy ---
    def train_lbfgs(self, maxiter=50000, maxcor=50):
        Xc = tf.convert_to_tensor(self.Collo, dtype=tf.float32)
        WALL = tf.convert_to_tensor(self.WALL, dtype=tf.float32)
        INLET = tf.convert_to_tensor(self.INLET, dtype=tf.float32)
        OUTLET = tf.convert_to_tensor(self.OUTLET, dtype=tf.float32)

        # pack/unpack weights
        def pack():
            return np.concatenate([v.numpy().reshape(-1) for v in self.net.trainable_variables]).astype(np.float64)

        def unpack(flat):
            idx = 0
            for v in self.net.trainable_variables:
                shape = v.shape
                size = int(np.prod(shape))
                v.assign(flat[idx:idx+size].reshape(shape).astype(np.float32))
                idx += size

        def loss_and_grad(flat):
            unpack(flat)
            with tf.GradientTape() as tape:
                loss, *_ = self.loss_fn(Xc, WALL, INLET, OUTLET)
            grads = tape.gradient(loss, self.net.trainable_variables)
            grad_flat = np.concatenate([g.numpy().reshape(-1) for g in grads]).astype(np.float64)
            return float(loss.numpy()), grad_flat

        x0 = pack()

        def callback(xk):
            l, _ = loss_and_grad(xk)
            self.loss_rec.append(l)
            total2, lf2, lw2, lin2, lout2 = self.loss_fn(Xc, WALL, INLET, OUTLET)
            self.loss_total.append(float(total2.numpy()))
            self.loss_pde.append(float(lf2.numpy()))
            self.loss_wall.append(float(lw2.numpy()))
            self.loss_inlet.append(float(lin2.numpy()))
            self.loss_outlet.append(float(lout2.numpy()))
            print("L-BFGS loss:", l)

        res = scipy.optimize.minimize(
            fun=lambda w: loss_and_grad(w),
            x0=x0,
            jac=True,
            method="L-BFGS-B",
            callback=callback,
            options={"maxiter": maxiter, "maxcor": maxcor, "ftol": np.finfo(float).eps}
        )
        unpack(res.x)
        return res

    # --- Save/Load (TF2 style) ---
    def save(self, path="uvNN_tf2"):
        self.net.save_weights(path)

    def load(self, path="uvNN_tf2"):
        # build once
        dummy = tf.zeros((1,2), dtype=tf.float32)
        _ = self.net(dummy)
        self.net.load_weights(path)

    # --- predict u,v,p (and optionally stress) ---

    def predict(self, x_star, y_star):
        x = tf.convert_to_tensor(x_star.astype(np.float32))
        y = tf.convert_to_tensor(y_star.astype(np.float32))
    
        with tf.GradientTape(persistent=True) as tape:
            tape.watch([x, y])
            psi, p, _, _, _ = self.net_psips(x, y)
    
        u = tape.gradient(psi, y)
        v = -tape.gradient(psi, x)
        del tape
    
        return u.numpy(), v.numpy(), p.numpy()

    def predict_all(self, x_star, y_star):
        x = tf.convert_to_tensor(x_star.astype(np.float32))
        y = tf.convert_to_tensor(y_star.astype(np.float32))
    
        with tf.GradientTape(persistent=True) as tape:
            tape.watch([x, y])
            psi, p, s11, s22, s12 = self.net_psips(x, y)
    
        u = tape.gradient(psi, y)
        v = -tape.gradient(psi, x)
        del tape
    
        return u.numpy(), v.numpy(), p.numpy(), s11.numpy(), s22.numpy(), s12.numpy()


## 4. Problem Setup

In [ ]:
# Domain
lb = np.array([0.0, 0.0])
ub = np.array([1.1, 0.41])
L, H = 1.1, 0.41
uv_layers = [2] + 8*[40] + [5]

# Cylinder
xc, yc, r = 0.2, 0.2, 0.05

# ---- Boundary points ----
Nw, Ni, No, Nc = 441, 201, 201, 251
wall_up = np.hstack([L*lhs(1, Nw), H*np.ones((Nw,1))])
wall_lw = np.hstack([L*lhs(1, Nw), np.zeros((Nw,1))])

U_max = 1.0
y_in  = H * lhs(1, Ni)
x_in  = np.zeros((Ni, 1))
u_in  = 4*U_max*y_in*(H-y_in)/H**2
INLET = np.hstack([x_in, y_in, u_in, np.zeros_like(u_in)])

OUTLET = np.hstack([L*np.ones((No,1)), H*lhs(1, No)])

theta_c = 2*np.pi*lhs(1, Nc)
CYLD    = np.hstack([xc + r*np.cos(theta_c), yc + r*np.sin(theta_c)])
WALL    = np.vstack([CYLD, wall_up, wall_lw])

# ---- Collocation points ----
XY_c        = lb + (ub - lb) * lhs(2, 40000)
XY_c_refine = np.array([0.1, 0.1]) + np.array([0.2, 0.2]) * lhs(2, 10000)
XY_c        = np.vstack([XY_c, XY_c_refine])
XY_c        = DelCylPT(XY_c, xc=xc, yc=yc, r=r)
XY_c        = np.vstack([XY_c, WALL, CYLD, OUTLET, INLET[:,0:2]])
print('Collocation points:', XY_c.shape[0])

fig, ax = plt.subplots(figsize=(8, 2.5))
ax.set_aspect('equal')
ax.scatter(XY_c[:,0], XY_c[:,1], s=0.5, alpha=0.1, label='Interior')
ax.scatter(WALL[:,0], WALL[:,1], s=4, alpha=0.5, label='Wall')
ax.scatter(OUTLET[:,0], OUTLET[:,1], s=6, alpha=0.6, label='Outlet')
ax.scatter(INLET[:,0], INLET[:,1], s=6, alpha=0.6, label='Inlet')
ax.legend(fontsize=7, loc='upper right')
ax.set_title('Collocation & boundary points')
plt.tight_layout(); plt.show()

## 5. Training

**Stage 1 — Adam** (10 000 iterations, lr = 5 × 10⁻⁴):  
Provides a good initial solution close to the loss landscape minimum.

**Stage 2 — L-BFGS-B** (up to 50 000 iters per round):  
Second-order quasi-Newton method for fine convergence.  
Run multiple rounds to continue refining from the saved checkpoint.

In [ ]:
model = PINN_laminar_flow_TF2(
    XY_c, INLET, OUTLET, WALL, uv_layers, lb, ub,
    rho=1.0, mu=0.02, bc_weight=2.0)

t0 = time.time()
model.train_adam(iters=10000, lr=5e-4, print_every=200)
n_adam = len(model.loss_rec)
print(f'Adam done in {time.time()-t0:.1f} s')

## 6. Evaluation helpers (run once)

In [ ]:
from scipy.interpolate import griddata

def evaluate_line(model, x_ref, y_ref, u_ref, v_ref, p_ref,
                  y0=0.205, n=300, xc=0.2, yc=0.2, r=0.05, xmax=1.1):
    """Compare PINN vs reference on horizontal line y=y0, report L2 errors."""
    x_line = np.linspace(0.0, xmax, n)[:, None]
    y_line = y0 * np.ones_like(x_line)
    mask   = np.sqrt((x_line - xc)**2 + (y_line - yc)**2)[:, 0] >= r
    xl, yl = x_line[mask], y_line[mask]

    u_p, v_p, p_p = model.predict(xl, yl)
    pts_ref = np.hstack([x_ref, y_ref])

    def _interp(vals):
        return griddata(pts_ref, vals.ravel(), np.hstack([xl, yl]), method="linear")

    u_r = _interp(u_ref); v_r = _interp(v_ref); p_r = _interp(p_ref)

    def _l2(pred, ref):
        return np.linalg.norm(pred.ravel() - ref.ravel()) / (np.linalg.norm(ref.ravel()) + 1e-12)

    print(f"  L2 u: {_l2(u_p, u_r):.4e}  |  L2 v: {_l2(v_p, v_r):.4e}  |  L2 p: {_l2(p_p, p_r):.4e}")

    fig, axes = plt.subplots(1, 3, figsize=(12, 3))
    for ax, pred, ref, lbl in zip(axes,
                                   [u_p, v_p, p_p], [u_r, v_r, p_r],
                                   [r"$u$", r"$v$", r"$p$"]):
        ax.plot(xl, pred, label="PINN")
        ax.plot(xl, ref, "--", label="Ref")
        ax.set_xlabel("x"); ax.set_ylabel(lbl)
        ax.legend(fontsize=8); ax.grid(alpha=0.4)
    fig.suptitle(f"Line comparison at y = {y0}")
    plt.tight_layout(); plt.show()


def plot_loss(model, n_adam=10000):
    """Semilogy plot of total loss with Adam/L-BFGS marker."""
    loss = np.array(model.loss_rec)
    plt.figure(figsize=(7, 4))
    plt.semilogy(loss, "k-", lw=1.2)
    if n_adam > 0 and n_adam < len(loss):
        plt.axvline(n_adam, color="r", ls="--", lw=1.5, label="Adam → L-BFGS")
        plt.legend()
    plt.xlabel("Iteration"); plt.ylabel("Loss")
    plt.title("Training Convergence")
    plt.grid(True, which="both", ls="--", alpha=0.4)
    plt.tight_layout(); plt.show()


def evaluate_field(model, x_ref, y_ref, u_ref, v_ref, p_ref,
                   L=1.1, H=0.41, xc=0.2, yc=0.2, r=0.05):
    """Full-field scatter comparison: PINN vs reference."""
    xg = np.linspace(0, L, 251); yg = np.linspace(0, H, 101)
    Xg, Yg = np.meshgrid(xg, yg)
    xs = Xg.ravel()[:, None]; ys = Yg.ravel()[:, None]
    mask = np.sqrt((xs - xc)**2 + (ys - yc)**2)[:, 0] >= r
    xs, ys = xs[mask], ys[mask]
    u_p, v_p, p_p = model.predict(xs, ys)
    field_ref  = [x_ref, y_ref, u_ref, v_ref, p_ref]
    field_pinn = [xs, ys, u_p, v_p, p_p]
    postProcess(0, L, 0, H, field_ref, field_pinn, s=3, alpha=0.5)


## 7. Round 1 — L-BFGS + checkpoint

In [ ]:
res1 = model.train_lbfgs(maxiter=50000)
model.save('uvNN_tf2_weights_1st.weights.h5')

plot_loss(model, n_adam)
x_ref, y_ref, u_ref, v_ref, p_ref = preprocess_mat('FluentSol.mat')
evaluate_field(model, x_ref, y_ref, u_ref, v_ref, p_ref)
evaluate_line(model, x_ref, y_ref, u_ref, v_ref, p_ref)

## 8. Round 2 — L-BFGS continued

In [ ]:
res2 = model.train_lbfgs(maxiter=50000)
model.save('uvNN_tf2_weights_2nd.weights.h5')

plot_loss(model, n_adam)
evaluate_field(model, x_ref, y_ref, u_ref, v_ref, p_ref)
evaluate_line(model, x_ref, y_ref, u_ref, v_ref, p_ref)

## 9. Round 3 — L-BFGS continued

In [ ]:
res3 = model.train_lbfgs(maxiter=50000)
model.save('uvNN_tf2_weights_3rd.weights.h5')

plot_loss(model, n_adam)
evaluate_field(model, x_ref, y_ref, u_ref, v_ref, p_ref)
evaluate_line(model, x_ref, y_ref, u_ref, v_ref, p_ref)

## 10. Round 4 — L-BFGS continued

In [ ]:
res4 = model.train_lbfgs(maxiter=50000)
model.save('uvNN_tf2_weights_4th.weights.h5')

plot_loss(model, n_adam)
evaluate_field(model, x_ref, y_ref, u_ref, v_ref, p_ref)
evaluate_line(model, x_ref, y_ref, u_ref, v_ref, p_ref)

## 11. Pressure Distribution on Cylinder Surface

In [ ]:
import matplotlib.pyplot as plt
from scipy.interpolate import griddata
from matplotlib.patches import Circle, FancyArrowPatch

def pressure_on_cylinder(theta, xc, yc, r, p_func=None, ref_data=None):
    x = xc + r * np.cos(theta)
    y = yc + r * np.sin(theta)
    if p_func is not None:
        return p_func(x[:, None], y[:, None]).reshape(-1)
    if ref_data is not None:
        x_r, y_r, p_r = ref_data
        pts_r = np.hstack([x_r.reshape(-1,1), y_r.reshape(-1,1)])
        pts_q = np.hstack([x.reshape(-1,1),   y.reshape(-1,1)])
        p = griddata(pts_r, p_r.ravel(), pts_q, method="linear")
        if np.any(np.isnan(p)):
            p2 = griddata(pts_r, p_r.ravel(), pts_q, method="nearest")
            p = np.where(np.isnan(p), p2, p)
        return p.reshape(-1)
    raise ValueError("Provide p_func or ref_data.")


def plot_pressure_distribution(theta, p_pred, p_ref=None, ylim=(-0.5, 4.0)):
    fig, ax = plt.subplots(figsize=(8, 4.5), dpi=120)
    ax.plot(theta, p_pred, lw=2.5, label="PINN")
    if p_ref is not None:
        ax.plot(theta, p_ref, lw=2.5, ls=":", label="Reference")
    ax.set_xlim(0, 2*np.pi); ax.set_ylim(*ylim)
    ax.set_xlabel(r"$\theta$ (rad)", fontsize=14)
    ax.set_ylabel("Pressure", fontsize=14)
    ax.set_xticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
    ax.set_xticklabels([r"$0$", r"$\pi/2$", r"$\pi$", r"$3\pi/2$", r"$2\pi$"])
    ax.legend(fontsize=12, frameon=False); ax.grid(alpha=0.4)
    plt.tight_layout(); plt.show()


xc, yc, r = 0.2, 0.2, 0.05
theta = np.linspace(0, 2*np.pi, 361)
p_pred_cyl = pressure_on_cylinder(theta, xc, yc, r,
    p_func=lambda x, y: model.predict(x, y)[2])
x_ref, y_ref, u_ref, v_ref, p_ref = preprocess_mat("FluentSol.mat")
p_ref_cyl  = pressure_on_cylinder(theta, xc, yc, r,
    ref_data=(x_ref, y_ref, p_ref))
plot_pressure_distribution(theta, p_pred_cyl, p_ref_cyl, ylim=(-0.5, 4.0))


## 12. Save Results

In [ ]:
results = {
    "loss_history": model.loss_rec,
    "theta": theta,
    "p_pred": p_pred_cyl,
    "p_ref":  p_ref_cyl,
    "meta": {
        "rho": 1.0, "mu": 0.02,
        "layers": uv_layers, "bc_weight": 2.0,
        "description": "Steady 2D cylinder laminar flow, mixed PINN (TF2)"
    }
}
with open("pinn_cylinder_results.pkl", "wb") as f:
    pickle.dump(results, f)
print("Saved pinn_cylinder_results.pkl")
